# 04 - Modelagem
Split out-of-time: treino < 2022-07, validação 2022-07..2023-03, teste >= 2023-04. Baseline: regressão logística; modelo: LightGBM.

In [1]:
import sys; sys.path.insert(0, '..')
import warnings; warnings.filterwarnings('ignore')
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
pd.set_option('display.width', 200, 'display.max_columns', 50)


In [2]:
from src.train import *
X, y, cli, data = carregar(); tr, va, te = split(data)
print('n:', tr.sum(), va.sum(), te.sum(), '| taxa:', y[tr].mean().round(3), y[va].mean().round(3), y[te].mean().round(3))

n: 60520 17780 25729 | taxa: 0.158 0.112 0.123


In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from src.evaluate import metricas
num, cat = X.select_dtypes('number').columns, X.select_dtypes('object').columns
Xl = X.copy(); Xl[num] = Xl[num].clip(Xl.loc[tr, num].quantile(.01), Xl.loc[tr, num].quantile(.99), axis=1)
lr = make_pipeline(ColumnTransformer([('n', make_pipeline(SimpleImputer(strategy='median'), StandardScaler()), num),
  ('c', make_pipeline(SimpleImputer(strategy='constant', fill_value='na'), OneHotEncoder(handle_unknown='ignore', min_frequency=100)), cat)]),
  LogisticRegression(max_iter=500, C=0.1))
lr.fit(Xl[tr], y[tr])
print('LR  teste OOT:', metricas(y[te], lr.predict_proba(Xl[te])[:,1]))

LR  teste OOT: {'auc': 0.6886794669498066, 'auc_ic95': (np.float64(0.6808142193374032), np.float64(0.6984974115945368)), 'gini': 0.3773589338996133, 'ks': 0.2783646844766722, 'brier': 0.10193482056412048, 'taxa': 0.12258540946014225}


In [4]:
m, cats, r, (Xc, yc, tr, va, te, p) = rodar()
print('LGBM teste OOT:', r, '| árvores:', m.best_iteration_)

LGBM teste OOT: {'auc': 0.7272352989506549, 'auc_ic95': (np.float64(0.7189402109437594), np.float64(0.7367705792391372)), 'gini': 0.4544705979013097, 'ks': 0.33846013183701756, 'brier': 0.09682792687028946, 'taxa': 0.12258540946014225, 'n_clientes_novos': 7955, 'auc_clientes_novos': 0.6181929091266718, 'auc_com_historico': 0.7334612797832766} | árvores: 259
